# Comparação de Modelos Whisper

Este notebook tem como objetivo comparar diferentes modelos do OpenAI Whisper em termos de:
- **Latência**: Tempo necessário para transcrever o áudio.
- **Qualidade**: O texto transcrito resultante.

Vamos testar os modelos: `tiny`, `base`, `small`, `medium` (e `large` opcionalmente).

In [1]:
import whisper
import time
import pandas as pd
import os
import sys

# Adicionar pasta src ao path para importar funções
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from extrair_audio import baixar_audio_youtube, transcrever_arquivo

from IPython.display import display, Audio

## 1. Preparação do Áudio do podcast

Vamos garantir que temos um arquivo de áudio para os testes. Se o arquivo não existir, baixaremos um vídeo curto do YouTube.

In [3]:
AUDIO_DIR = "../data"
AUDIO_FILE = "audio_receita.mp3"
AUDIO_PATH = os.path.join(AUDIO_DIR, AUDIO_FILE)

# URL de um vídeo para teste
# Substitua por outra URL se desejar testar com outro áudio
YOUTUBE_URL = "https://www.youtube.com/watch?v=4bkrcEUBr-k"

if not os.path.exists(AUDIO_PATH):
    print("⏳ Baixando áudio para teste...")
    try:
        baixar_audio_youtube(YOUTUBE_URL, AUDIO_PATH, quiet=False)
        print(f"✅ Áudio baixado com sucesso: {AUDIO_PATH}")
    except Exception as e:
        print(f"❌ Erro ao baixar áudio: {e}")
else:
    print(f"✅ Arquivo de áudio já existe: {AUDIO_PATH}")

⏳ Baixando áudio para teste...
⏳ Baixando áudio com yt-dlp...
[youtube] Extracting URL: https://www.youtube.com/watch?v=4bkrcEUBr-k
[youtube] 4bkrcEUBr-k: Downloading webpage
[youtube] 4bkrcEUBr-k: Downloading android sdkless player API JSON
[youtube] 4bkrcEUBr-k: Downloading tv client config
[youtube] 4bkrcEUBr-k: Downloading tv player API JSON
[youtube] 4bkrcEUBr-k: Downloading web safari player API JSON
[youtube] 4bkrcEUBr-k: Downloading player a644926e-main
[youtube] 4bkrcEUBr-k: Downloading m3u8 information
[info] 4bkrcEUBr-k: Downloading 1 format(s): 251-1
[download] Sleeping 4.00 seconds as required by the site...
[download] Destination: ..\data\audio_receita.mp3
[download] 100% of    9.20MiB in 00:00:01 at 5.43MiB/s   
✅ Áudio baixado em: ../data\audio_receita.mp3
✅ Áudio baixado com sucesso: ../data\audio_receita.mp3


## 2. Configuração dos Modelos

Defina quais modelos deseja comparar. 
Modelos disponíveis: `tiny`, `base`, `small`, `medium`, `large`.

> **Nota**: Modelos maiores (`medium`, `large`) requerem mais VRAM (GPU) ou RAM (CPU) e levarão mais tempo.

In [4]:
modelos_para_testar = ["tiny", "base", "small", "medium"] # Adicione "large" se tiver hardware suficiente
resultados = []

## 3. Execução do Teste

Iteramos sobre os modelos, carregamos cada um, transcrevemos o áudio e medimos o tempo.

In [5]:
print(f"Iniciando testes com o arquivo: {AUDIO_PATH}")

for modelo_nome in modelos_para_testar:
    print(f"\n--- Testando modelo: {modelo_nome} ---")
    
    try:
        # Usando a função importada de extrair_audio.py
        # Ela retorna um dict com 'text', 'load_time', 'transcribe_time'
        resultado = transcrever_arquivo(AUDIO_PATH, modelo=modelo_nome)
        
        print(f"Modelo carregado em {resultado['load_time']:.2f}s")
        print(f"Transcrição concluída em {resultado['transcribe_time']:.2f}s")
        
        # Salvar Resultados
        resultados.append({
            "Modelo": modelo_nome,
            "Tempo Carregamento (s)": round(resultado['load_time'], 2),
            "Tempo Transcrição (s)": round(resultado['transcribe_time'], 2),
            "Texto": resultado["text"].strip()
        })
        
    except Exception as e:
        print(f"Erro ao testar modelo {modelo_nome}: {e}")
        resultados.append({
            "Modelo": modelo_nome,
            "Tempo Carregamento (s)": None,
            "Tempo Transcrição (s)": None,
            "Texto": f"Erro: {str(e)}"
        })

Iniciando testes com o arquivo: ../data\audio_receita.mp3

--- Testando modelo: tiny ---
⏳ Carregando modelo Whisper (tiny)...
⏳ Transcrevendo áudio...


c:\Users\HugoLeonardo\Documents\GitHub\Data-Science---Python\Mentoria_NLP\projeto_uv\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Transcrição concluída.
Modelo carregado em 0.74s
Transcrição concluída em 66.34s

--- Testando modelo: base ---
⏳ Carregando modelo Whisper (base)...
⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 0.77s
Transcrição concluída em 103.13s

--- Testando modelo: small ---
⏳ Carregando modelo Whisper (small)...
⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 2.77s
Transcrição concluída em 316.82s

--- Testando modelo: medium ---
⏳ Carregando modelo Whisper (medium)...
⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 9.71s
Transcrição concluída em 818.59s


## 4. Análise dos Resultados

Abaixo apresentamos a tabela comparativa e os textos gerados.

In [6]:
df_resultados = pd.DataFrame(resultados)

# Exibir tabela de tempos
print("\n=== Comparação de Latência ===")
display(df_resultados[["Modelo", "Tempo Carregamento (s)", "Tempo Transcrição (s)"]])

# Exibir comparação de textos
print("\n=== Comparação de Qualidade (Texto) ===")
for index, row in df_resultados.iterrows():
    print(f"\n>>> Modelo: {row['Modelo']}")
    print(f"Tempo: {row['Tempo Transcrição (s)']}s")
    print("Texto:")
    print(row['Texto'][:500] + "..." if len(row['Texto']) > 500 else row['Texto'])
    print("-" * 80)


=== Comparação de Latência ===


,Modelo,Tempo Carregamento (s),Tempo Transcrição (s)
0,tiny,0.74,66.34
1,base,0.77,103.13
2,small,2.77,316.82
3,medium,9.71,818.59



=== Comparação de Qualidade (Texto) ===

>>> Modelo: tiny
Tempo: 66.34s
Texto:
Olá, tudo bem? Meu nome é Jainha Figueira, eu sou cháfio de cozinha e hoje eu vou fazer um doce italiano maravilhoso que se chama Zabayone. É uma coisa muito fácil de se fazer, tá? Aqui no Brasil, a gente pode chamar de uma gemada. Mas é uma gemada x, que é uma gemada? Ela é elaborada, porque a gente vai usar nela um vinho licoroso. Aqui hoje eu estou com um vinho mais stala, mas também você pode usar um vincento, pode usar um chereço, pode usar um vinho do porto, mas como uma esfa suria chá. A ...
--------------------------------------------------------------------------------

>>> Modelo: base
Tempo: 103.13s
Texto:
Olá, tudo bem? Meu nome é Jair Nifigueira, eu sou o chef de cozinha e hoje eu vou fazer um doce italiano maravilhoso que se chama Zabayone. É uma coisa muito fácil de se fazer, tá? Aqui no Brasil a gente pode chamar de uma gemada, mas é uma gemada chique, uma gemada, ela é elaborada, porque a g

In [7]:
# Opcional: Salvar resultados em CSV
df_resultados.to_csv("../data/comparacao_whisper_receita.csv", index=False)
print("Resultados salvos em data/comparacao_whisper_receita.csv")

Resultados salvos em data/comparacao_whisper_receita.csv
